# Notebook 03: Improved Denial-Risk Model Comparison

This notebook improves the less-leaky denial-risk model by testing multiple machine learning algorithms.

The goal is to compare models using the same processed CMS modeling dataset and select the best production candidate for the DenialOps AI dashboard.

The target variable is `denial_risk_proxy`, which was created in Notebook 01 using reimbursement-friction signals. Since this is a proxy label and not a true hospital denial outcome, the model should be interpreted as a reimbursement-risk scoring system, not a final denial decision engine.

## 1. Imports

This section imports the libraries needed for data loading, preprocessing, model training, evaluation, threshold tuning, visualization, and model saving.

In [ ]:
# Import libraries for data loading, preprocessing, modeling, evaluation, visualization, and model saving

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Model comparison environment ready")

## 2. Load Modeling Dataset

This section loads the final 250,000-row modeling dataset created in Notebook 01.

The dataset already contains selected CMS provider-service fields, engineered reimbursement features, and the proxy denial-risk target.

In [ ]:
# Load the final modeling dataset created in Notebook 01

path = "../data/processed/denial_risk_modeling_sample_250k.csv"

df = pd.read_csv(path)

print(df.shape)
df.head()

In [ ]:
# Check the dataset columns and datatypes before modeling

df.info()

In [ ]:
# Check the target class balance for the proxy denial-risk label

df["denial_risk_proxy"].value_counts()

In [ ]:
# Check the target class balance as percentages

df["denial_risk_proxy"].value_counts(normalize=True)

## 3. Safer Feature Engineering

This section creates safer features for the less-leaky model.

The direct proxy-label input features are not used:

- `charge_to_allowed_ratio`
- `payment_to_charge_ratio`
- `payment_to_allowed_ratio`
- `services_per_beneficiary`

Instead, this notebook uses provider, service, volume, and reimbursement amount fields. Log transformations are added to reduce skew in high-volume and high-dollar fields.

In [ ]:
# Create log-transformed numeric features to reduce skew in high-volume and high-dollar columns

df["log_total_services"] = np.log1p(df["total_services"])
df["log_total_beneficiaries"] = np.log1p(df["total_beneficiaries"])
df["log_total_beneficiary_day_services"] = np.log1p(df["total_beneficiary_day_services"])

df["log_avg_submitted_charge"] = np.log1p(df["avg_submitted_charge"])
df["log_avg_medicare_allowed_amount"] = np.log1p(df["avg_medicare_allowed_amount"])
df["log_avg_medicare_payment_amount"] = np.log1p(df["avg_medicare_payment_amount"])
df["log_avg_medicare_standardized_amount"] = np.log1p(df["avg_medicare_standardized_amount"])

df.head()

In [ ]:
# Group rare HCPCS codes to reduce high-cardinality noise

top_hcpcs_codes = (
    df["hcpcs_code"]
    .value_counts()
    .head(500)
    .index
)

df["hcpcs_code_grouped"] = np.where(
    df["hcpcs_code"].isin(top_hcpcs_codes),
    df["hcpcs_code"],
    "OTHER"
)

df["hcpcs_code_grouped"].value_counts().head()

## 4. Define Features and Target

This section defines the model input features and the target variable.

The feature set excludes direct proxy-label input features to reduce leakage.

In [ ]:
# Define the less-leaky feature set for model comparison

features = [
    "provider_state",
    "provider_type",
    "medicare_participating",
    "hcpcs_code_grouped",
    "is_drug",
    "place_of_service",
    "total_beneficiaries",
    "total_services",
    "total_beneficiary_day_services",
    "avg_submitted_charge",
    "avg_medicare_allowed_amount",
    "avg_medicare_payment_amount",
    "avg_medicare_standardized_amount",
    "log_total_services",
    "log_total_beneficiaries",
    "log_total_beneficiary_day_services",
    "log_avg_submitted_charge",
    "log_avg_medicare_allowed_amount",
    "log_avg_medicare_payment_amount",
    "log_avg_medicare_standardized_amount"
]

target = "denial_risk_proxy"

features_df = df[features].copy()
target_series = df[target].copy()

print(features_df.shape)
print(target_series.shape)

In [ ]:
# Separate categorical and numeric features for preprocessing

categorical_features = [
    "provider_state",
    "provider_type",
    "medicare_participating",
    "hcpcs_code_grouped",
    "is_drug",
    "place_of_service"
]

numeric_features = [
    "total_beneficiaries",
    "total_services",
    "total_beneficiary_day_services",
    "avg_submitted_charge",
    "avg_medicare_allowed_amount",
    "avg_medicare_payment_amount",
    "avg_medicare_standardized_amount",
    "log_total_services",
    "log_total_beneficiaries",
    "log_total_beneficiary_day_services",
    "log_avg_submitted_charge",
    "log_avg_medicare_allowed_amount",
    "log_avg_medicare_payment_amount",
    "log_avg_medicare_standardized_amount"
]

print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))

## 5. Train/Test Split

This section splits the data into training and test sets.

Stratification is used so the target class distribution remains consistent between the training and test data.

In [ ]:
# Split the dataset into training and test sets while preserving class balance

features_train, features_test, target_train, target_test = train_test_split(
    features_df,
    target_series,
    test_size=0.2,
    random_state=42,
    stratify=target_series
)

print(features_train.shape)
print(features_test.shape)
print(target_train.value_counts(normalize=True))
print(target_test.value_counts(normalize=True))

## 6. Preprocessing Pipelines

This section creates reusable preprocessing steps.

- Logistic Regression uses one-hot encoding and scaled numeric features.
- Random Forest, Extra Trees, and XGBoost use one-hot encoding and passthrough numeric features.
- Gradient Boosting and HistGradientBoosting use ordinal encoding to avoid large dense one-hot matrices.

In [ ]:
# Create preprocessing for Logistic Regression using one-hot encoding and scaled numeric features

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

linear_preprocessor

In [ ]:
# Create preprocessing for sparse-compatible tree models using one-hot encoding and passthrough numeric features

onehot_tree_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

onehot_tree_preprocessor

In [ ]:
# Create preprocessing for dense boosting models using ordinal encoding and passthrough numeric features

ordinal_tree_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

ordinal_tree_preprocessor

## 7. Evaluation Function

This section creates one function to evaluate every model consistently.

The metrics used are:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC

For this project, recall is especially important because the app is meant to flag risky claims for review.

In [ ]:
# Create a reusable evaluation function for all models

def evaluate_model(model_name, pipeline, features_test, target_test):
    target_pred = pipeline.predict(features_test)
    target_proba = pipeline.predict_proba(features_test)[:, 1]
    
    results = {
        "model": model_name,
        "accuracy": accuracy_score(target_test, target_pred),
        "precision": precision_score(target_test, target_pred),
        "recall": recall_score(target_test, target_pred),
        "f1_score": f1_score(target_test, target_pred),
        "roc_auc": roc_auc_score(target_test, target_proba)
    }
    
    return results, target_pred, target_proba

In [ ]:
# Create containers to store model results, trained pipelines, and prediction probabilities

model_results = []
trained_pipelines = {}
model_probabilities = {}

## 8. Model 1: Logistic Regression

Logistic Regression is used as a simple, interpretable baseline.

It is not expected to be the strongest model, but it gives a useful comparison point.

In [ ]:
# Build the Logistic Regression pipeline

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", linear_preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

logistic_pipeline

In [ ]:
# Train the Logistic Regression model

logistic_pipeline.fit(features_train, target_train)

print("Logistic Regression training complete")

In [ ]:
# Evaluate the Logistic Regression model

logistic_regression_results, logistic_regression_pred, logistic_regression_proba = evaluate_model(
    "Logistic Regression",
    logistic_pipeline,
    features_test,
    target_test
)

model_results.append(logistic_regression_results)
trained_pipelines["Logistic Regression"] = logistic_pipeline
model_probabilities["Logistic Regression"] = logistic_regression_proba

logistic_regression_results

## 9. Model 2: Random Forest

Random Forest is the original less-leaky baseline model family.

This improved version uses the updated feature set with log features and grouped HCPCS codes.

In [ ]:
# Build the Random Forest Improved pipeline

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", onehot_tree_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=16,
            min_samples_split=20,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

random_forest_pipeline

In [ ]:
# Train the Random Forest Improved model

random_forest_pipeline.fit(features_train, target_train)

print("Random Forest Improved training complete")

In [ ]:
# Evaluate the Random Forest Improved model

random_forest_improved_results, random_forest_improved_pred, random_forest_improved_proba = evaluate_model(
    "Random Forest Improved",
    random_forest_pipeline,
    features_test,
    target_test
)

model_results.append(random_forest_improved_results)
trained_pipelines["Random Forest Improved"] = random_forest_pipeline
model_probabilities["Random Forest Improved"] = random_forest_improved_proba

random_forest_improved_results

## 10. Model 3: Extra Trees

Extra Trees is similar to Random Forest but adds more randomness when building trees.

It can sometimes perform better on noisy tabular datasets.

In [ ]:
# Build the Extra Trees pipeline

extra_trees_pipeline = Pipeline(
    steps=[
        ("preprocessor", onehot_tree_preprocessor),
        ("model", ExtraTreesClassifier(
            n_estimators=200,
            max_depth=16,
            min_samples_split=20,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

extra_trees_pipeline

In [ ]:
# Train the Extra Trees model

extra_trees_pipeline.fit(features_train, target_train)

print("Extra Trees training complete")

In [ ]:
# Evaluate the Extra Trees model

extra_trees_results, extra_trees_pred, extra_trees_proba = evaluate_model(
    "Extra Trees",
    extra_trees_pipeline,
    features_test,
    target_test
)

model_results.append(extra_trees_results)
trained_pipelines["Extra Trees"] = extra_trees_pipeline
model_probabilities["Extra Trees"] = extra_trees_proba

extra_trees_results

## 11. Model 4: Gradient Boosting

Gradient Boosting builds trees sequentially, where each new tree tries to correct errors from previous trees.

This model uses ordinal-encoded categorical features to keep memory usage manageable.

In [ ]:
# Build the Gradient Boosting pipeline

gradient_boosting_pipeline = Pipeline(
    steps=[
        ("preprocessor", ordinal_tree_preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        ))
    ]
)

gradient_boosting_pipeline

In [ ]:
# Train the Gradient Boosting model

gradient_boosting_pipeline.fit(features_train, target_train)

print("Gradient Boosting training complete")

In [ ]:
# Evaluate the Gradient Boosting model

gradient_boosting_results, gradient_boosting_pred, gradient_boosting_proba = evaluate_model(
    "Gradient Boosting",
    gradient_boosting_pipeline,
    features_test,
    target_test
)

model_results.append(gradient_boosting_results)
trained_pipelines["Gradient Boosting"] = gradient_boosting_pipeline
model_probabilities["Gradient Boosting"] = gradient_boosting_proba

gradient_boosting_results

## 12. Model 5: HistGradientBoosting

HistGradientBoosting is a faster gradient boosting implementation in scikit-learn.

It is often a strong choice for larger tabular datasets.

In [ ]:
# Build the HistGradientBoosting pipeline

hist_gradient_pipeline = Pipeline(
    steps=[
        ("preprocessor", ordinal_tree_preprocessor),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            random_state=42
        ))
    ]
)

hist_gradient_pipeline

In [ ]:
# Train the HistGradientBoosting model

hist_gradient_pipeline.fit(features_train, target_train)

print("HistGradientBoosting training complete")

In [ ]:
# Evaluate the HistGradientBoosting model

histgradientboosting_results, histgradientboosting_pred, histgradientboosting_proba = evaluate_model(
    "HistGradientBoosting",
    hist_gradient_pipeline,
    features_test,
    target_test
)

model_results.append(histgradientboosting_results)
trained_pipelines["HistGradientBoosting"] = hist_gradient_pipeline
model_probabilities["HistGradientBoosting"] = histgradientboosting_proba

histgradientboosting_results

## 13. Model 6: XGBoost

XGBoost is a powerful gradient boosting model commonly used for structured tabular data.

It is included because this project is a tabular classification problem with mixed categorical and numeric features.

In [ ]:
# Calculate scale_pos_weight for XGBoost to account for class imbalance

negative_count = (target_train == 0).sum()
positive_count = (target_train == 1).sum()

scale_pos_weight = negative_count / positive_count

scale_pos_weight

In [ ]:
# Build the XGBoost pipeline

xgboost_pipeline = Pipeline(
    steps=[
        ("preprocessor", onehot_tree_preprocessor),
        ("model", XGBClassifier(
            n_estimators=250,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

xgboost_pipeline

In [ ]:
# Train the XGBoost model

xgboost_pipeline.fit(features_train, target_train)

print("XGBoost training complete")

In [ ]:
# Evaluate the XGBoost model

xgboost_results, xgboost_pred, xgboost_proba = evaluate_model(
    "XGBoost",
    xgboost_pipeline,
    features_test,
    target_test
)

model_results.append(xgboost_results)
trained_pipelines["XGBoost"] = xgboost_pipeline
model_probabilities["XGBoost"] = xgboost_proba

xgboost_results

## 14. Model Comparison

This section compares all trained models using the same test set.

The best model should balance recall, F1 score, and ROC AUC. Since this is a denial-risk review tool, recall matters because missing high-risk claims is costly.

In [ ]:
# Create a model comparison dataframe sorted by ROC AUC

model_results_df = pd.DataFrame(model_results)

model_results_df = model_results_df.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

model_results_df

In [ ]:
# Plot model ROC AUC comparison

plt.figure(figsize=(10, 6))

sns.barplot(
    data=model_results_df,
    x="roc_auc",
    y="model"
)

plt.title("Model Comparison by ROC AUC")
plt.xlabel("ROC AUC")
plt.ylabel("Model")
plt.show()

In [ ]:
# Plot model F1 score comparison

plt.figure(figsize=(10, 6))

sns.barplot(
    data=model_results_df.sort_values("f1_score", ascending=False),
    x="f1_score",
    y="model"
)

plt.title("Model Comparison by F1 Score")
plt.xlabel("F1 Score")
plt.ylabel("Model")
plt.show()

In [ ]:
# Plot model recall comparison

plt.figure(figsize=(10, 6))

sns.barplot(
    data=model_results_df.sort_values("recall", ascending=False),
    x="recall",
    y="model"
)

plt.title("Model Comparison by Recall")
plt.xlabel("Recall")
plt.ylabel("Model")
plt.show()

## 15. Select Production Candidate

This section selects the production candidate based on the highest F1 score, using ROC AUC as a tie-breaker.

F1 score is used because this project needs a balance between precision and recall.

In [ ]:
# Select the production candidate using F1 score first and ROC AUC second

production_row = (
    model_results_df
    .sort_values(by=["f1_score", "roc_auc"], ascending=False)
    .iloc[0]
)

production_model_name = production_row["model"]
production_pipeline = trained_pipelines[production_model_name]
production_proba = model_probabilities[production_model_name]

print(f"Selected production candidate: {production_model_name}")
production_row

## 16. Threshold Tuning

The default classification threshold is 0.50.

For denial-risk scoring, a lower threshold may be useful because the app is creating a review queue. The goal is to catch more high-risk claims, even if some false positives are included.

In [ ]:
# Test multiple probability thresholds for the production model

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    threshold_pred = (production_proba >= threshold).astype(int)
    
    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(target_test, threshold_pred),
        "recall": recall_score(target_test, threshold_pred),
        "f1_score": f1_score(target_test, threshold_pred)
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

In [ ]:
# Plot precision, recall, and F1 score across thresholds

plt.figure(figsize=(10, 6))

sns.lineplot(data=threshold_results_df, x="threshold", y="precision", marker="o", label="Precision")
sns.lineplot(data=threshold_results_df, x="threshold", y="recall", marker="o", label="Recall")
sns.lineplot(data=threshold_results_df, x="threshold", y="f1_score", marker="o", label="F1 Score")

plt.title("Production Model Threshold Tuning")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
# Select the best threshold based on the highest F1 score

best_threshold_row = threshold_results_df.sort_values(
    by="f1_score",
    ascending=False
).iloc[0]

best_threshold = best_threshold_row["threshold"]

best_threshold_row

## 17. Final Production Model Evaluation

This section evaluates the production model using the selected threshold.

In [ ]:
# Generate final production predictions using the selected threshold

production_pred = (production_proba >= best_threshold).astype(int)

production_accuracy = accuracy_score(target_test, production_pred)
production_precision = precision_score(target_test, production_pred)
production_recall = recall_score(target_test, production_pred)
production_f1 = f1_score(target_test, production_pred)
production_roc_auc = roc_auc_score(target_test, production_proba)

production_metrics = {
    "model": production_model_name,
    "threshold": best_threshold,
    "accuracy": production_accuracy,
    "precision": production_precision,
    "recall": production_recall,
    "f1_score": production_f1,
    "roc_auc": production_roc_auc
}

production_metrics

In [ ]:
# Display the final classification report for the production model

print(classification_report(target_test, production_pred))

In [ ]:
# Plot the final production confusion matrix

cm = confusion_matrix(target_test, production_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Low Risk", "High Risk"],
    yticklabels=["Low Risk", "High Risk"]
)

plt.title(f"Confusion Matrix - {production_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 18. Save Production Artifacts

This section saves the selected production model, threshold, model comparison results, and final test predictions.

These artifacts will be used by the Streamlit dashboard.

In [ ]:
# Save the production model pipeline

joblib.dump(
    production_pipeline,
    "../models/production_denial_risk_model.joblib"
)

print("Production model saved successfully")

In [ ]:
# Save the selected production threshold

joblib.dump(
    best_threshold,
    "../models/production_threshold.joblib"
)

print("Production threshold saved successfully")

In [ ]:
# Save the model comparison results

model_results_df.to_csv(
    "../reports/model_comparison_results.csv",
    index=False
)

print("Model comparison results saved successfully")

In [ ]:
# Save the final production metrics

production_metrics_df = pd.DataFrame([production_metrics])

production_metrics_df.to_csv(
    "../reports/production_model_metrics.csv",
    index=False
)

production_metrics_df

In [ ]:
# Save production test predictions for later error analysis

production_predictions_df = features_test.copy()
production_predictions_df["actual_denial_risk_proxy"] = target_test.values
production_predictions_df["predicted_denial_risk_proxy"] = production_pred
production_predictions_df["predicted_high_risk_probability"] = production_proba

production_predictions_df.to_csv(
    "../reports/production_test_predictions.csv",
    index=False
)

print("Production test predictions saved successfully")

## 19. Conclusion

This notebook compared multiple less-leaky models for denial-risk scoring using the same CMS-derived modeling dataset.

The production model was selected based on model performance and business fit. Since the app is designed to create a review queue for billing teams, recall and F1 score are more important than accuracy alone.

The saved production model and threshold will be used in the Streamlit dashboard.